# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [1]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [4]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

2026/05/26 12:24:02 INFO mlflow.tracking.fluent: Experiment with name 'check-localhost-connection' does not exist. Creating a new experiment.


🏃 View run delightful-trout-172 at: http://localhost:5000/#/experiments/1/runs/1e7517fd991a422b904a05e2dfb209f0
🧪 View experiment at: http://localhost:5000/#/experiments/1


Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [5]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2026/05/26 12:31:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/26 12:31:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'sk-learn-random-forest-reg-model'.
2026/05/26 12:31:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 1


🏃 View run legendary-bird-808 at: http://localhost:5000/#/experiments/1/runs/5b9edc4ad8834047a25f224d8d1e425a
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '1' of model 'sk-learn-random-forest-reg-model'.


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [7]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.49980732  44.5541425   41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.44675489 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [8]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [9]:
# %pip install hyperopt
%pip install -U git+https://github.com/hyperopt/hyperopt

  Cloning https://github.com/hyperopt/hyperopt to c:\users\oscar\appdata\local\temp\pip-req-build-oxjrcs9i
  Resolved https://github.com/hyperopt/hyperopt to commit c49ad148201c81c6ad1b43730ef4d0912734aa37
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for hyperopt: filename=hyperopt-0.3.0-py3-none-any.whl size=973207 sha256=3f982b7ecb6f013c1aef9aa1a10bfbcabb6ac080c8741d84e72392bdf077afeb
  Stored in directory: C:\Users\oscar\AppData\Local\Temp\pip-ephem-wheel-cache-logdb8sr\wheels\ad\e0\dc\af4d21315718e63bc2e53ded682ca11817b02cb72d3935cf0d
Successfully built hyperopt
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/hyperopt/hyperopt 'C:\Users\oscar\AppData\Local\Temp\pip-req-build-oxjrcs9i'

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [11]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [12]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

2026/05/26 12:55:30 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779792930106, experiment_id='2', last_update_time=1779792930106, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>

In [13]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

  0%|          | 0/8 [00:00<?, ?trial/s, best loss=?]WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 17s 399ms/step - loss: 39.2899 - root_mean_squared_error: 6.2682
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 32.9945 - root_mean_squared_error: 5.7441 - val_loss: 29.0555 - val_root_mean_squared_error: 5.3903

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 28.3905 - root_mean_squared_error: 5.3283
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 25.5070 - root_mean_squared_error: 5.0504 - val_loss: 22.3542 - val_root_mean_squared_error: 4.7280

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 19.8749 - root_mean_squared_error: 4.4581
46/46 

2026/05/26 12:55:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run dashing-crab-301 at: http://localhost:5000/#/experiments/2/runs/aa260e6e4cc547448443cd352d388ea1

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 14s 312ms/step - loss: 42.5777 - root_mean_squared_error: 6.5252
44/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 31.2972 - root_mean_squared_error: 5.5701   
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 21.7384 - root_mean_squared_error: 4.6624 - val_loss: 7.2125 - val_root_mean_squared_error: 2.6856

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 7.1710 - root_mean_squared_error: 2.6779
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9804 - root_mean_squared_error: 2.2246 
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.8940 - root_mean_squared_error: 1.9733 - val_loss: 2.5614 - val_root_mean_squared_error: 1.6004

Epoch 3/3      

2026/05/26 12:55:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run painted-mule-996 at: http://localhost:5000/#/experiments/2/runs/7022d1985d6341c6b9267620a921652f

🧪 View experiment at: http://localhost:5000/#/experiments/2                  

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 13s 301ms/step - loss: 38.8447 - root_mean_squared_error: 6.2326
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 37.6985 - root_mean_squared_error: 6.1399 - val_loss: 37.0546 - val_root_mean_squared_error: 6.0873

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 36.6317 - root_mean_squared_error: 6.0524
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 36.0240 - root_mean_squared_error: 6.0020 - val_loss: 35.4165 - val_root_mean_squared_error: 5.9512

Epoch 3/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 34.9885 - root_mean_squared_error: 5.9151
46/4

...

🏃 View run indecisive-trout-145 at: http://localhost:5000/#/experiments/2/runs/aae3b0fd646945b2aba86f73044b539e

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 14s 316ms/step - loss: 38.8625 - root_mean_squared_error: 6.2340
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 38.9134 - root_mean_squared_error: 6.2381 - val_loss: 36.2065 - val_root_mean_squared_error: 6.0172

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 34.9822 - root_mean_squared_error: 5.9146
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 33.2186 - root_mean_squared_error: 5.7636 - val_loss: 31.0053 - val_root_mean_squared_error: 5.5682

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 31.2893 - root_mean_squared_error: 5.5

2026/05/26 12:56:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run loud-sloth-644 at: http://localhost:5000/#/experiments/2/runs/e6be42b41cbb4de9b0d2dca96037244b

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [01:29<00:00, 11.17s/trial, best loss: 0.7415153384208679]

2026/05/26 12:56:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.0842018650292149), 'momentum': np.float64(0.08783540973402537)}
Best eval rmse: 0.7415153384208679
🏃 View run loud-shad-668 at: http://localhost:5000/#/experiments/2/runs/bd6f211615904312b1a11412ef061b06
🧪 View experiment at: http://localhost:5000/#/experiments/2


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [14]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 287ms/step - loss: 31.6970 - root_mean_squared_error: 5.6300
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.9387 - root_mean_squared_error: 2.8176 - val_loss: 2.0852 - val_root_mean_squared_error: 1.4440

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 1.4426 - root_mean_squared_error: 1.2011
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.6850 - root_mean_squared_error: 1.2981 - val_loss: 1.5502 - val_root_mean_squared_error: 1.2451

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 1.2088 - root_mean_squared_error: 1.0995
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3293 - root_mean_squared_error: 1.1530 - val_loss: 1.3083 - val_root_mean_squared_error: 1.1438

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 1.1220 - root_mean_squared_error: 1.0592
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

2026/05/26 12:57:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run capricious-hog-287 at: http://localhost:5000/#/experiments/2/runs/0bf85d91faa74cebb6d2f558881e5210

🧪 View experiment at: http://localhost:5000/#/experiments/2

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 286ms/step - loss: 32.1246 - root_mean_squared_error: 5.6679
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.7802 - root_mean_squared_error: 2.7893 - val_loss: 1.9057 - val_root_mean_squared_error: 1.3805

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 2.4225 - root_mean_squared_error: 1.5564
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.6091 - root_mean_squared_error: 1.2685 - val_loss: 1.4995 - val_root_mean_squared_error: 1.2246

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 1.6717 - root_mean_squared_error: 1.2929
46/46 ━━━━━━━━━━━━━━━━━

2026/05/26 12:57:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run respected-snail-762 at: http://localhost:5000/#/experiments/2/runs/f5ef6973b2464144a33157dc80f51860

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 24s 555ms/step - loss: 37.2909 - root_mean_squared_error: 6.1066
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 15.6478 - root_mean_squared_error: 3.9557 - val_loss: 3.0744 - val_root_mean_squared_error: 1.7534

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 2.9254 - root_mean_squared_error: 1.7104
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3974 - root_mean_squared_error: 1.5483 - val_loss: 1.9795 - val_root_mean_squared_error: 1.4070

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 1.0981 - root_mean_squared_error: 1.0479
46

...

🏃 View run rogue-newt-949 at: http://localhost:5000/#/experiments/2/runs/9278d3fe2c7d4aaa806b04172c0e0684

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 281ms/step - loss: 29.4269 - root_mean_squared_error: 5.4247
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 26.2212 - root_mean_squared_error: 5.1207 - val_loss: 22.1709 - val_root_mean_squared_error: 4.7086

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 20.0950 - root_mean_squared_error: 4.4827
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19.2557 - root_mean_squared_error: 4.3881 - val_loss: 16.1737 - val_root_mean_squared_error: 4.0217

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 17.3099 - root_mean_squared_error: 4.1605
46

2026/05/26 12:58:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run marvelous-pug-616 at: http://localhost:5000/#/experiments/2/runs/e437c67de1f34d9fba8c1052f3f3f052

🧪 View experiment at: http://localhost:5000/#/experiments/2                   

100%|██████████| 8/8 [01:25<00:00, 10.68s/trial, best loss: 0.7366804480552673]

2026/05/26 12:58:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.08525840690127226), 'momentum': np.float64(0.36420042873048886)}
Best eval rmse: 0.7366804480552673
🏃 View run big-pig-796 at: http://localhost:5000/#/experiments/2/runs/271eb5e542cc4a549132e0c2739e9ade
🧪 View experiment at: http://localhost:5000/#/experiments/2


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [15]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

ConnectionError: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000138C8342410>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))

In [ ]:
response.content